In [1]:
import pandas as pd, os, datetime
import numpy as np
from scipy import stats
from sklearn.cluster import KMeans
from sklearn import metrics
import numpy as np
from sklearn.metrics import silhouette_samples, silhouette_score
import geopandas as gpd

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import seaborn as sns
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
sdate, edate = '2009-07-01','2024-06-30'

In [4]:
def jitter(group):
    duplicates = group.groupby(['lat', 'lon']).cumcount()
    
    # Jitter function: add a small random offset
    np.random.seed(42)  # For reproducibility
    jitter_strength = 0.05 # Adjust as needed; degrees latitude/longitude
    
    group['lat_jittered'] = group['lat'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates
    group['lon_jittered'] = group['lon'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates

    return group

In [5]:
def process_group_daily(grp, gen_fpath, hw_tseries, start_date=None, end_date=None):
    """
    Processes a single group of generator data.

    Parameters:
        grp (pd.DataFrame): A single group from gen_details (e.g., from groupby('region')).
        gen_fpath (str): Path to directory containing CSV files named by DUID (e.g., DUID.csv).
        hw_tseries (pd.DataFrame): DataFrame with columns 'time', 'DUID', and data to merge.
        start_date (str): Start date for subsetting the time series.
        end_date (str): End date for subsetting the time series.

    Returns:
        pd.DataFrame: The merged result for the group.
    """
    safe_duids = [duid.replace("/", "_").replace("\\", "_") for duid in grp['DUID']]
    gen_locs = [f"{gen_fpath}/{duid}.csv" for duid in safe_duids]
    dfs = [pd.read_csv(fp,dtype='object') for fp in gen_locs if os.path.exists(fp)]
    print(f"Loaded {len(dfs)} CSV file(s) out of {len(gen_locs)} expected.")

    if not dfs:
        return None

    # This is in case of accidental mid-file headers
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # This is to correct the column types after removing header rows
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)
    
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    # Group by DUID and hourly time, sum TOTALMWh
    agg_func = {'TOTALMWh':'sum','TOTALCLEARED':'sum','AGCSTATUS':'max'}
    grouped = dfs.groupby(['DUID', pd.Grouper(freq='1d')]).agg(agg_func).reset_index()

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index(['time']).sort_index()
    hw_tseries = hw_tseries.loc[sdate:edate]
    hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

    merged = pd.merge_asof(
        grouped.sort_values(by=['time', 'DUID']),
        hw_tseries.sort_values(by=['time', 'DUID']),
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("1d"),
        direction='nearest'
    )

    merged = merged.dropna(how='all')

    return merged

In [6]:
def select_group(gen_details, state=None, ftype=None):
    if state is not None and ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[(gen_details['region'] == state) & (gen_details['fuel_source_primary'].isin(ftype))]
        else:
            groups = gen_details.groupby(['region', 'fuel_source_primary'])
            grp = groups.get_group((state, ftype))
    elif state is not None:
        groups = gen_details.groupby('region')
        grp = groups.get_group(state)
    elif ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[gen_details['fuel_source_primary'].isin(ftype)]
        else:
            groups = gen_details.groupby('fuel_source_primary')
            grp = groups.get_group(ftype)
    else:
        grp = gen_details

    return grp


In [7]:
def clean_df(df, gen_details): 
    df = df.merge(gen_details[['DUID', 'reg_cap_mw','technology_type_primary','fuel_source_primary','lat_jittered','lon_jittered', 'region']], on='DUID', how='left')
    df = df[~((df['TOTALMWh'] < 0))]
    mask = (df['fuel_source_primary'].isin([
            'Water', 'Natural Gas Pipeline', 'Black Coal', 'Coal Seam Methane',
            'Brown Coal', 'Diesel', 'Kerosene'
        ]) & (df['TOTALCLEARED'] <= 0))
    df.loc[mask, 'TOTALMWh'] = np.nan
    
    djf = [12, 1, 2]  # December, January, February
    df = df[df['time'].dt.month.isin(djf)]
 
    return df

In [8]:
def min_heatwave_days(df, min_days=20):
    """
    Filters the DataFrame to include only DUIDs with at least `min_days`
    of unique heatwave days (EHF_flag == 1).
    
    Parameters:
        df (pd.DataFrame): Input DataFrame with columns ['DUID', 'time', 'EHF_flag']
        min_days (int): Minimum number of unique heatwave days required

    Returns:
        pd.DataFrame: Filtered DataFrame
    """
    df = df.copy()
    df['time'] = pd.to_datetime(df['time'])
    df['date'] = df['time'].dt.date

    # Count unique heatwave days per DUID
    heatwave_days = (
        df[df['EHF_flag'] == 1]
        .groupby('DUID')['date']
        .nunique()
    )

    # Keep only DUIDs meeting the threshold
    valid_duids = heatwave_days[heatwave_days >= min_days].index
    
    return df[df['DUID'].isin(valid_duids)].copy()


In [9]:
def remove_wind_zero_duids(df, duid_col='DUID', value_col='TOTALMWh',
                          tech_col='fuel_source_primary', wind_label='Wind', threshold=5):
    """
    Remove *all rows* for wind DUIDs where the percentage of zeros 
    exceeds the given threshold. Non-wind DUIDs are left untouched.
    
    Parameters:
        df (pd.DataFrame): Input dataframe
        duid_col (str): Column name for grouping (default 'DUID')
        value_col (str): Column name with numeric values (default 'TOTALMWh')
        tech_col (str): Column that identifies technology type (default 'fuel_source_primary')
        wind_label (str): Label used for wind in tech_col (default 'Wind')
        threshold (float): Maximum allowed percentage of zeros (default 5)
    
    Returns:
        pd.DataFrame: Filtered dataframe
    """
    
    # work only on wind rows
    wind_df = df[df[tech_col] == wind_label]
    
    # calculate % of zeros per wind DUID
    percent_zeros = (
        wind_df.groupby(duid_col)[value_col]
               .apply(lambda x: (x == 0).sum() / len(x) * 100)
    )
    
    # keep DUIDs below threshold
    keep_duids = percent_zeros[percent_zeros <= threshold].index
    
    # filter wind df
    wind_filtered = wind_df[wind_df[duid_col].isin(keep_duids)]
    
    # keep all non-wind rows
    non_wind_df = df[df[tech_col] != wind_label]
    
    # combine and return
    return pd.concat([non_wind_df, wind_filtered], ignore_index=True)

In [10]:
gen_details = pd.read_csv(f"{nmap_path}/gen_info.csv")
gen_details = jitter(gen_details)

In [11]:
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")

In [12]:
info = select_group(gen_details, ftype=['Wind']).copy()
df = process_group_daily(info, gen_fpath, hw_tseries, sdate, edate)

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
df = clean_df(df, info)
df = remove_wind_zero_duids(df, threshold=40)
df = min_heatwave_days(df, 20)

In [ ]:
df2 = min_heatwave_days(df, 20)

In [ ]:
df = df.copy()
df['time'] = pd.to_datetime(df['time'])
df['date'] = df['time'].dt.date

# Count unique heatwave days per DUID
heatwave_days = (
    df[df['EHF_flag'] == 1]
    .groupby('DUID')['date']
    .nunique()
)
heatwave_days

In [ ]:
df2[df2['DUID']=='BANGOWF1']

In [ ]:
# Plot all wind farms
fig = px.scatter_map(
    df,
    lat='lat_jittered',
    lon='lon_jittered',
    hover_name='DUID',
    hover_data={
        'lat': False,
        'lon': False,
        'lat_jittered': False,
        'lon_jittered': False,
        'fuel_source_primary': True,
    },
    color='fuel_source_primary',
    
    zoom=3,
    map_style='carto-darkmatter'
)
fig.update_traces(marker=dict(size=5))

fig.update_layout(
    legend=dict(
        title="Fuel Type",
    ),
    title_text="Locations of Generation Units"
)

fig.show()

In [ ]:
df = info[info['DUID'].isin(df['DUID'])].copy()

In [ ]:
#Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")

#Project to UTM zone 56S (EPSG:32756). Adjust if further west.
gdf_utm = gdf.to_crs(epsg=32756)

# Extract x,y in meters
X = np.vstack([gdf_utm.geometry.x, gdf_utm.geometry.y]).T

# Step 3: Run KMeans
inertias = []
K_range = range(1, 20)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

# Step 4: Elbow plot
plt.plot(K_range, inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (within-cluster sum of squares)")
plt.title("Elbow Method with KMeans (UTM projection)")
plt.show()

In [ ]:
#Get cluster assignments
best_k = 6
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

In [ ]:
# Convert centroids back to lat/lon for plotting
centers_utm = kmeans.cluster_centers_
centers_gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(centers_utm[:,0], centers_utm[:,1]),
    crs="EPSG:32756"
)
centers_latlon = centers_gdf.to_crs(epsg=4326)
centers_df = pd.DataFrame({
    "lat": centers_latlon.geometry.y,
    "lon": centers_latlon.geometry.x,
    "cluster": [f"Centroid {i}" for i in range(len(centers_utm))]
})

df['cluster_str'] = df['cluster'].astype(str)

fig = px.scatter_map(
    df,
    lat="lat",
    lon="lon",
    hover_name="DUID",
    hover_data={"lat": False, "lon": False, "cluster": True},
    color="cluster_str",
    zoom=3,
    map_style="carto-positron"
)

# Add centroids with bright color and bigger size
fig.add_scattermap(
    lat=centers_df["lat"],
    lon=centers_df["lon"],
    mode="markers",
    marker=dict(size=7, color='white', symbol='star'), 
    text=centers_df["cluster"],
    hoverinfo="text",
    name="Cluster Centroids"
)

fig.update_layout(
    title="K-means clusters of NEM wind farms",
    legend_title_text="Cluster"
)

fig.update_layout(title="K-means clusters of NEM wind farms")
fig.show()

In [ ]:
# Compute silhouette scores for each point
sil_scores = silhouette_samples(X, df['cluster'])

# Add to your df
df['silhouette'] = sil_scores

# Average silhouette per cluster
for c in range(best_k):
    avg_score = df[df['cluster']==c]['silhouette'].mean()
    print(f"Cluster {c}: avg silhouette = {avg_score:.3f}")

# Plot silhouette distribution per cluster
plt.figure(figsize=(8,5))
for c in range(best_k):
    cluster_scores = df[df['cluster']==c]['silhouette']
    plt.hist(cluster_scores, bins=10, alpha=0.6, label=f'Cluster {c}')
plt.xlabel('Silhouette score')
plt.ylabel('Number of points')
plt.title('Silhouette Distribution per Cluster')
plt.legend()
plt.show()
